# 00b — Set STATUS to OK after validation

Run this notebook once, after you finish the validation. It needs no GPU.

1. It makes a backup copy of `luganda_eval_items.xlsx` in the same folder.
2. It changes every `TODO` in the STATUS columns to `OK`. It does not change rows that are `FIXED` or `DROP`, and it does not change any text.
3. It prints the new counts.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, datetime
from pathlib import Path
from openpyxl import load_workbook

PATH = Path("/content/drive/MyDrive/fatima technical challenge/luganda_eval_items.xlsx")
assert PATH.exists(), f"Workbook not found: {PATH}"

backup = PATH.with_name(f"luganda_eval_items_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}.xlsx")
shutil.copy2(PATH, backup)
print("Backup:", backup)

wb = load_workbook(PATH)
for sheet in ["minimal_pairs", "paragraphs", "variation"]:
    ws = wb[sheet]
    headers = [c.value for c in ws[1]]
    col = headers.index("STATUS") + 1
    counts, changed = {}, 0
    for r in range(2, ws.max_row + 1):
        if ws.cell(row=r, column=1).value in (None, ""):
            continue
        cell = ws.cell(row=r, column=col)
        value = str(cell.value or "").strip().upper()
        if value in ("", "TODO"):
            cell.value = "OK"
            value = "OK"
            changed += 1
        counts[value] = counts.get(value, 0) + 1
    print(f"{sheet}: changed {changed} rows to OK | counts now: {counts}")
wb.save(PATH)
print("Saved:", PATH)